# 04 — Validate Marketing Results

**Goal:** Independently recompute the main numbers in Notebook 03 from `data/processed/`, then compare them with saved output tables.



- **PASS**: the calculation agrees with the expected figure or saved table.
- **FAIL**: investigate before publishing.
- **INFO**: a known data limitation; not automatically an error.

The expected headline numbers below are rounded figures previously reported in the analysis. These checks verify arithmetic and consistency, **not** whether campaign attribution proves causation.

## 1. Imports and source files

In [9]:
import numpy as np
import pandas as pd
from IPython.display import display
from src.config import PROCESSED_DATA_DIR, TABLES_DIR

orders = pd.read_csv(PROCESSED_DATA_DIR / "orders.csv", parse_dates=["order_date"])
items = pd.read_csv(PROCESSED_DATA_DIR / "order_items.csv")
products = pd.read_csv(PROCESSED_DATA_DIR / "products.csv")
campaigns = pd.read_csv(PROCESSED_DATA_DIR / "campaigns.csv", parse_dates=["start_date", "end_date"])
customers = pd.read_csv(PROCESSED_DATA_DIR / "customers.csv", parse_dates=["signup_date"])

for col in ["in_reporting_period", "is_revenue_eligible", "flag_placeholder_date"]:
    if col in orders:
        orders[col] = orders[col].astype("string").str.lower().map({"true": True, "false": False})

checks = []
def check(name, actual, expected, tolerance=0):
    passed = pd.notna(actual) and pd.notna(expected) and abs(float(actual) - float(expected)) <= tolerance
    checks.append({"check": name, "actual": actual, "expected": expected, "difference": float(actual)-float(expected), "status": "PASS" if passed else "FAIL"})

def check_true(name, passed, detail=""):
    checks.append({"check": name, "actual": detail, "expected": "True", "difference": np.nan, "status": "PASS" if bool(passed) else "FAIL"})

print("Processed data loaded.")

Processed data loaded.


## 2. Structural and reporting-period checks

Check that duplicates were removed and that eligible orders really are completed orders in the intended time window.

In [10]:
check_true("Unique order IDs", orders["order_id"].is_unique)
check_true("Unique customer IDs", customers["customer_id"].is_unique)
check_true("Unique campaign IDs", campaigns["campaign_id"].is_unique)
check_true("Unique product IDs", products["product_id"].is_unique)

period = orders.loc[orders["in_reporting_period"].eq(True)].copy()
sales = orders.loc[orders["is_revenue_eligible"].eq(True)].copy()
check_true("Revenue-eligible statuses are completed", sales["status"].eq("completed").all())
check_true("Revenue-eligible totals nonnegative", sales["total_amount"].ge(0).all())
check_true("Revenue-eligible orders in reporting period", sales["order_date"].between("2024-07-01", "2026-06-30").all())
check_true("Eligible orders are also period orders", sales["order_id"].isin(period["order_id"]).all())
check_true("Sales campaign IDs exist", sales["campaign_id"].dropna().isin(campaigns["campaign_id"]).all())
check_true("Order item product IDs exist", items["product_id"].dropna().isin(products["product_id"]).all())
check_true("Sales order IDs have items", sales["order_id"].isin(items["order_id"]).all())

print(f"Reporting-period orders: {len(period):,}")
print(f"Revenue-eligible orders: {len(sales):,}")

Reporting-period orders: 14,910
Revenue-eligible orders: 12,811


## 3. Independently recompute headline metrics

Amounts are compared with **rounded** previously reported results, so a tolerance is used where necessary.

In [11]:
attributed = sales[sales["campaign_id"].notna()].copy()
revenue = sales["total_amount"].sum()
attributed_revenue = attributed["total_amount"].sum()
budgets = campaigns["budget_eur"].sum()
summary = pd.Series({
    "completed_revenue_eur": revenue,
    "completed_orders": sales["order_id"].nunique(),
    "purchasing_customers": sales["customer_id"].nunique(),
    "average_order_value_eur": revenue / sales["order_id"].nunique(),
    "attributed_revenue_eur": attributed_revenue,
    "campaign_budgets_eur": budgets,
    "revenue_roas": attributed_revenue / budgets,
    "attributed_revenue_share_pct": 100 * attributed_revenue / revenue,
})
display(summary.to_frame("recomputed"))

check("Completed revenue ≈ €4,215,636", revenue, 4_215_636, tolerance=1)
check("Completed orders", sales["order_id"].nunique(), 12_811)
check("Purchasing customers", sales["customer_id"].nunique(), 3_006)
check("Attributed revenue ≈ €569,493", attributed_revenue, 569_493, tolerance=1)
check("Campaign budgets", budgets, 407_500)
check("Revenue ROAS ≈ 1.40", attributed_revenue / budgets, 1.40, tolerance=0.01)
check("Attributed revenue share ≈ 13.51%", 100 * attributed_revenue / revenue, 13.51, tolerance=0.01)

,recomputed
completed_revenue_eur,4.215636e+06
completed_orders,1.281100e+04
purchasing_customers,3.006000e+03
average_order_value_eur,3.290638e+02
attributed_revenue_eur,5.694931e+05
campaign_budgets_eur,4.075000e+05
revenue_roas,1.397529e+00
attributed_revenue_share_pct,1.350906e+01


## 4. Independently recalculate product contribution

The order cost formula is `sum(quantity × product unit cost)`. **Do not drop repeated item rows here**, because Notebook 03 retained them. Check missing unit costs before trusting contribution figures.

In [12]:
item_costs = items.merge(products[["product_id", "unit_cost_eur"]], on="product_id", how="left", validate="many_to_one")
missing_cost_rows = item_costs["unit_cost_eur"].isna().sum()
check_true("No missing product unit costs", missing_cost_rows == 0, f"Missing rows: {missing_cost_rows}")
item_costs["line_cost_eur"] = item_costs["quantity"] * item_costs["unit_cost_eur"]
cost_per_order = item_costs.groupby("order_id", as_index=False)["line_cost_eur"].sum()
contrib_sales = sales.merge(cost_per_order, on="order_id", how="left", validate="one_to_one")
check_true("Every eligible order has calculated product costs", contrib_sales["line_cost_eur"].notna().all())
contrib_sales["contribution_eur"] = contrib_sales["total_amount"] - contrib_sales["line_cost_eur"]
contrib_attributed = contrib_sales[contrib_sales["campaign_id"].notna()].copy()
contrib_total = contrib_attributed["contribution_eur"].sum()
net_total = contrib_total - budgets
print(f"Attributed product contribution: €{contrib_total:,.2f}")
print(f"Attributed contribution after budgets: €{net_total:,.2f}")
check("Attributed contribution ≈ €116,745", contrib_total, 116_745, tolerance=2)
check("Contribution after budgets ≈ −€290,755", net_total, -290_755, tolerance=2)

Attributed product contribution: €116,745.35
Attributed contribution after budgets: €-290,754.65


## 5. Recompute campaign metrics and compare to exported CSV

This is the most useful consistency test: every campaign’s revenue, contribution and budget must match the independently rebuilt calculation.

In [13]:
recalc = contrib_attributed.groupby("campaign_id").agg(
    attributed_orders=("order_id", "nunique"),
    attributed_revenue_eur=("total_amount", "sum"),
    attributed_contribution_eur=("contribution_eur", "sum"),
).reset_index()
recalc = campaigns[["campaign_id", "campaign_name", "budget_eur"]].merge(recalc, on="campaign_id", how="left", validate="one_to_one")
for col in ["attributed_orders", "attributed_revenue_eur", "attributed_contribution_eur"]:
    recalc[col] = recalc[col].fillna(0)
recalc["roas"] = recalc["attributed_revenue_eur"] / recalc["budget_eur"].replace(0, np.nan)
recalc["contribution_roas"] = recalc["attributed_contribution_eur"] / recalc["budget_eur"].replace(0, np.nan)
recalc["contribution_after_budget_eur"] = recalc["attributed_contribution_eur"] - recalc["budget_eur"]

saved_path = TABLES_DIR / "campaign_performance.csv"
if saved_path.exists():
    saved = pd.read_csv(saved_path)
    compared = recalc.merge(saved, on="campaign_id", how="outer", suffixes=("_recalc", "_saved"), indicator=True)
    check_true("Campaign IDs match exported file", compared["_merge"].eq("both").all())
    for metric in ["budget_eur", "attributed_orders", "attributed_revenue_eur", "attributed_contribution_eur", "roas", "contribution_roas", "contribution_after_budget_eur"]:
        a, b = metric+"_recalc", metric+"_saved"
        if b in compared and a in compared:
            delta = (compared[a] - compared[b]).abs()
            check_true(f"Saved campaign {metric} matches", delta.le(0.02).all(), f"Maximum absolute difference: {delta.max():.4f}")
        else:
            check_true(f"Saved campaign {metric} exists", False, f"Missing column in saved export: {metric}")
    display(compared[["campaign_id", "campaign_name_recalc", "contribution_after_budget_eur_recalc", "contribution_after_budget_eur_saved"]])
else:
    check_true("Campaign performance CSV exists", False, str(saved_path))

check("Sum of campaign budgets", recalc["budget_eur"].sum(), budgets, tolerance=0.01)
check("Sum of campaign attributed revenue", recalc["attributed_revenue_eur"].sum(), attributed_revenue, tolerance=0.01)
check("Sum of campaign net contribution", recalc["contribution_after_budget_eur"].sum(), net_total, tolerance=0.01)

,campaign_id,campaign_name_recalc,contribution_after_budget_eur_recalc,contribution_after_budget_eur_saved
0,C01,Back to Uni 2024,-10502.18,-10502.18
1,C02,Black Friday 2024,-41884.60,-41884.60
2,C03,Xmas Gifts 2024,3301.41,3301.41
3,C04,New Year Fitness,-6347.96,-6347.96
4,C05,Spring Home Refresh,-315.85,-315.85
5,C06,Easter Family Deals,3239.47,3239.47
6,C07,Mother's Day,-9815.20,-9815.20
7,C08,Loyalty Points x2,18581.59,18581.59
8,C09,Summer Mega Sale,-194911.58,-194911.58
9,C10,Back to Uni 2025,-7347.53,-7347.53


## 6. Check campaign status rates independently

In [16]:
period_attributed = period[period["campaign_id"].notna()].copy()
status_counts = (period_attributed.groupby(["campaign_id", "status"])["order_id"]
                 .nunique().unstack(fill_value=0))
status_counts["total_orders"] = period_attributed.groupby("campaign_id")["order_id"].nunique()
status_counts["return_rate_pct"] = 100 * status_counts.get("returned", 0) / status_counts["total_orders"]
status_counts["cancellation_rate_pct"] = 100 * status_counts.get("cancelled", 0) / status_counts["total_orders"]
display(status_counts[["total_orders", "return_rate_pct", "cancellation_rate_pct"]].round(2))
if saved_path.exists():
    saved = pd.read_csv(saved_path).set_index("campaign_id")
    for metric in ["return_rate_pct", "cancellation_rate_pct"]:
        if metric in saved:
            delta = (status_counts[metric] - saved[metric]).abs()
            check_true(f"Saved {metric} matches", delta.fillna(0).le(0.02).all(), f"Max absolute difference: {delta.max():.4f}")
        else:
            check_true(f"Saved {metric} exists", False, "Export campaign table again after computing rates")

status,total_orders,return_rate_pct,cancellation_rate_pct
campaign_id,,,
C01,140,12.86,2.86
C02,90,8.89,4.44
C03,125,9.60,6.40
C04,172,12.21,2.91
C05,198,9.60,3.54
C06,65,7.69,6.15
C07,83,14.46,3.61
C08,209,9.57,3.35
C09,685,11.82,3.80


## 7. Check sensitivity formula and first-purchase order counts

Sensitivity assumes a fraction of *attributed product contribution* is incremental. It does not estimate true incrementality. First-purchase status is based on observed completed-order history.

In [17]:
recalc["required_incremental_share_pct"] = np.where(
    recalc["attributed_contribution_eur"] > 0,
    100 * recalc["budget_eur"] / recalc["attributed_contribution_eur"],
    np.inf,
)
for share in (0.25, 0.5, 0.75, 1.0):
    recalc[f"net_{int(share*100)}pct_eur"] = recalc["attributed_contribution_eur"]*share - recalc["budget_eur"]
check_true("Contribution ROAS at 1× matches 100% scenario break-even", np.all((recalc["contribution_roas"] >= 1) == (recalc["net_100pct_eur"] >= -0.01)))
display(recalc[["campaign_name", "required_incremental_share_pct", "net_25pct_eur", "net_50pct_eur", "net_100pct_eur"]].round(2))

history = orders.loc[
    orders["status"].eq("completed") & orders["order_date"].notna()
    & ~orders["flag_placeholder_date"].fillna(False),
    ["customer_id", "order_date"]
].copy()
first_date = history.groupby("customer_id")["order_date"].min().rename("first_purchase_date")
first_orders = attributed.merge(first_date, on="customer_id", how="left", validate="many_to_one")
first_orders["first_purchase"] = first_orders["order_date"].dt.normalize().eq(first_orders["first_purchase_date"].dt.normalize())
first_by_campaign = first_orders.groupby("campaign_id").agg(
    attributed_completed_orders=("order_id", "nunique"),
    first_purchase_orders=("first_purchase", "sum"),
).reset_index().merge(campaigns[["campaign_id", "campaign_name"]], on="campaign_id", how="left")
first_by_campaign["first_purchase_pct"] = 100 * first_by_campaign["first_purchase_orders"] / first_by_campaign["attributed_completed_orders"]
display(first_by_campaign.sort_values("first_purchase_orders", ascending=False))

summer = first_by_campaign.loc[first_by_campaign["campaign_name"].eq("Summer Mega Sale")]
if len(summer) == 1:
    check("Summer Mega Sale first-purchase orders", summer["first_purchase_orders"].iloc[0], 452)
    check("Summer Mega Sale first-purchase share ≈ 78.3%", summer["first_purchase_pct"].iloc[0], 78.3, tolerance=0.1)
else:
    check_true("Summer Mega Sale campaign found", False)

,campaign_name,required_incremental_share_pct,net_25pct_eur,net_50pct_eur,net_100pct_eur
0,Back to Uni 2024,191.34,-19125.54,-16251.09,-10502.18
1,Black Friday 2024,784.90,-46471.15,-44942.30,-41884.60
2,Xmas Gifts 2024,64.51,-3674.65,-1349.30,3301.41
3,New Year Fitness,154.48,-15086.99,-12173.98,-6347.96
4,Spring Home Refresh,102.15,-11328.96,-7657.92,-315.85
5,Easter Family Deals,58.14,-2565.13,-630.26,3239.47
6,Mother's Day,334.54,-12953.80,-11907.60,-9815.20
7,Loyalty Points x2,13.90,2395.40,7790.80,18581.59
8,Summer Mega Sale,inf,-187477.90,-189955.79,-194911.58
9,Back to Uni 2025,144.12,-19836.88,-15673.76,-7347.53


,campaign_id,attributed_completed_orders,first_purchase_orders,campaign_name,first_purchase_pct
8,C09,577,452,Summer Mega Sale,78.336222
0,C01,118,71,Back to Uni 2024,60.169492
11,C12,195,41,Spring Clean 2026,21.025641
3,C04,145,37,New Year Fitness,25.517241
9,C10,206,37,Back to Uni 2025,17.961165
4,C05,172,30,Spring Home Refresh,17.441860
7,C08,180,28,Loyalty Points x2,15.555556
2,C03,105,25,Xmas Gifts 2024,23.809524
1,C02,78,22,Black Friday 2024,28.205128
6,C07,68,17,Mother's Day,25.000000


## 8. Known-answer unit tests — can the formulas handle controlled examples?

These tiny examples have **hand-calculated answers** and test the production calculation pattern, not the NovoStar aggregate totals. They deliberately include multiple line items, quantity > 1, and a returned order.

In [18]:
# Small hand-checkable fixtures. Expected O1 cost = 2×30 + 1×20 = €80.
fixture_products = pd.DataFrame({"product_id": ["P1", "P2"], "unit_cost_eur": [30.0, 20.0]})
fixture_items = pd.DataFrame({"order_id": ["O1", "O1", "O2"], "product_id": ["P1", "P2", "P2"], "quantity": [2, 1, 1]})
fixture_orders = pd.DataFrame({"order_id": ["O1", "O2"], "status": ["completed", "returned"], "total_amount": [100.0, 50.0], "campaign_id": ["C1", "C1"]})
fixture_budget = 10.0

# The same merge/groupby expression used in Notebook 03, applied to known inputs.
fi = fixture_items.merge(fixture_products, on="product_id", how="left", validate="many_to_one")
fi["cost"] = fi["quantity"] * fi["unit_cost_eur"]
fc = fi.groupby("order_id")["cost"].sum()
check("Known-answer: multiple-line O1 cost", fc["O1"], 80.0)
check("Known-answer: O2 cost", fc["O2"], 20.0)
fixture_completed = fixture_orders.query("status == 'completed'").copy()
fixture_completed["cost"] = fixture_completed["order_id"].map(fc)
fixture_completed["contribution"] = fixture_completed["total_amount"] - fixture_completed["cost"]
check("Known-answer: returned order excluded", len(fixture_completed), 1)
check("Known-answer: completed revenue", fixture_completed["total_amount"].sum(), 100)
check("Known-answer: contribution", fixture_completed["contribution"].sum(), 20)
check("Known-answer: revenue ROAS", 100 / fixture_budget, 10)
check("Known-answer: contribution ROAS", 20 / fixture_budget, 2)
check("Known-answer: after-budget contribution", 20 - fixture_budget, 10)
check("Known-answer: 50% scenario", 0.5 * 20 - fixture_budget, 0)
print("Known-answer checks registered.")

Known-answer checks registered.


## 9. Independent accounting identities and filter checks

These checks look for **population leaks or double counts**, using an independently rebuilt eligibility mask rather than trusting `is_revenue_eligible`.

In [19]:
# Independent definition from raw columns; compare record IDs against stored eligibility flags.
expected_eligible = (
    orders["order_date"].between("2024-07-01", "2026-06-30")
    & orders["status"].eq("completed")
    & orders["total_amount"].ge(0)
    & ~orders["flag_placeholder_date"].fillna(False)
)
check_true("Eligibility flag agrees with independently rebuilt conditions",
           orders["is_revenue_eligible"].fillna(False).eq(expected_eligible).all(),
           f"Disagreeing rows: {int(orders['is_revenue_eligible'].fillna(False).ne(expected_eligible).sum())}")

# No order should enter both partitions.
no_campaign = sales["campaign_id"].isna()
with_campaign = sales["campaign_id"].notna()
check("Partition covers all eligible orders", int(no_campaign.sum() + with_campaign.sum()), len(sales))
check("Attributed + unattributed revenue = completed revenue",
      sales.loc[no_campaign, "total_amount"].sum() + sales.loc[with_campaign, "total_amount"].sum(),
      sales["total_amount"].sum(), tolerance=0.01)

# Independent ledger identity, verified per order rather than only totals.
identity_error = (contrib_sales["total_amount"] - contrib_sales["line_cost_eur"] - contrib_sales["contribution_eur"]).abs()
check_true("Revenue minus cost equals contribution on EVERY order",
           identity_error.le(0.01).all(), f"Worst residual: {identity_error.max():.6f}")

# Ensure a left join didn't accidentally multiply orders.
check("Joining order costs preserves eligible row count", len(contrib_sales), len(sales))
check_true("Campaign budgets strictly positive", campaigns["budget_eur"].gt(0).all())


## 10. Manual spot-check worksheet — trace actual orders to source lines

Pick five real attributed orders, including a few from Summer Mega Sale. Inspect each order's raw line items and product costs. **The displayed `manual_expected_cost_eur` is a suggested arithmetic cross-check, not independent evidence**: manually multiply the visible quantities and costs for at least two orders and compare to the result. The row-level worksheet makes incorrect product joins, quantities, or unexpected repeated lines visible.

In [20]:
# Deterministic sample: 3 Summer Mega Sale orders and 2 from other campaigns.
summer_ids = sales.loc[sales["campaign_id"].eq("C09"), "order_id"].head(3).tolist()
other_ids = sales.loc[sales["campaign_id"].notna() & ~sales["campaign_id"].eq("C09"), "order_id"].head(2).tolist()
spot_ids = list(dict.fromkeys(summer_ids + other_ids))
spot_rows = items.loc[items["order_id"].isin(spot_ids)].merge(
    products[["product_id", "unit_cost_eur"]], on="product_id", how="left", validate="many_to_one"
)
spot_rows["line_cost_manual_guide"] = spot_rows["quantity"] * spot_rows["unit_cost_eur"]
print("Inspect line items and manually check quantity × unit cost:")
display(spot_rows[["order_id", "product_id", "quantity", "unit_cost_eur", "line_cost_manual_guide"]])
spot_totals = spot_rows.groupby("order_id")["line_cost_manual_guide"].sum()
spot_ledger = contrib_sales.loc[contrib_sales["order_id"].isin(spot_ids),
                                ["order_id", "campaign_id", "total_amount", "line_cost_eur", "contribution_eur"]].copy()
spot_ledger["cost_from_visible_items"] = spot_ledger["order_id"].map(spot_totals)
spot_ledger["cost_difference"] = spot_ledger["line_cost_eur"] - spot_ledger["cost_from_visible_items"]
display(spot_ledger)
check_true("Sampled order costs match visible item-level arithmetic",
           spot_ledger["cost_difference"].abs().le(0.01).all())
print("Human step: check at least two orders by hand and record any anomalies in your report.")

Inspect line items and manually check quantity × unit cost:


,order_id,product_id,quantity,unit_cost_eur,line_cost_manual_guide
0,O003069,P0111,3,49.99,149.97
1,O003069,P0098,1,2.64,2.64
2,O005949,P0044,1,70.61,70.61
3,O005949,P0126,1,76.27,76.27
4,O005949,P0136,2,29.49,58.98
5,O005949,P0136,1,29.49,29.49
6,O007006,P0065,2,95.19,190.38
7,O007006,P0072,1,14.76,14.76
8,O007606,P0136,1,29.49,29.49
9,O007606,P0026,1,101.70,101.70


,order_id,campaign_id,total_amount,line_cost_eur,contribution_eur,cost_from_visible_items,cost_difference
615,O003069,C01,190.56,152.61,37.95,152.61,0.0
616,O007006,C01,252.57,205.14,47.43,205.14,0.0
4868,O014378,C09,92.40,88.40,4.00,88.40,0.0
4873,O005949,C09,245.04,235.35,9.69,235.35,0.0
4876,O007606,C09,200.44,205.11,-4.67,205.11,0.0


Human step: check at least two orders by hand and record any anomalies in your report.


## 11. Robustness — can duplicate line items or unmatched order totals change campaign conclusions?

The data audit found exact repeated item lines and order total differences. We do **not** assume those lines are wrong. Instead, compare base estimates with a hypothetical version removing identical item lines. Report any materially different results as an uncertainty, not a correction.

In [21]:
def costs_from_lines(lines):
    joined = lines.merge(products[["product_id", "unit_cost_eur"]], on="product_id", how="left", validate="many_to_one")
    return (joined.assign(cost=joined["quantity"] * joined["unit_cost_eur"])
                  .groupby("order_id")["cost"].sum())

base_cost = costs_from_lines(items)
alternative_cost = costs_from_lines(items.drop_duplicates())
check("Base cost matches main aggregation", base_cost.sum(), item_costs["line_cost_eur"].sum(), tolerance=0.01)

robust = attributed[["order_id", "campaign_id", "total_amount"]].copy()
robust["base_cost"] = robust["order_id"].map(base_cost)
robust["deduped_item_cost"] = robust["order_id"].map(alternative_cost)
robust["base_contribution"] = robust["total_amount"] - robust["base_cost"]
robust["alternative_contribution"] = robust["total_amount"] - robust["deduped_item_cost"]
comparison = robust.groupby("campaign_id").agg(
    base_contribution=("base_contribution", "sum"),
    alternative_contribution=("alternative_contribution", "sum"),
).join(campaigns.set_index("campaign_id")[["campaign_name", "budget_eur"]])
comparison["base_after_budget"] = comparison["base_contribution"] - comparison["budget_eur"]
comparison["alternative_after_budget"] = comparison["alternative_contribution"] - comparison["budget_eur"]
comparison["difference_eur"] = comparison["alternative_after_budget"] - comparison["base_after_budget"]
display(comparison[["campaign_name", "base_after_budget", "alternative_after_budget", "difference_eur"]].sort_values("difference_eur", ascending=False))
print("Order-item duplicate rows:", items.duplicated().sum())
print("These are scenario differences, not evidence that deleting item rows is correct.")

# Order-item net selling price vs recorded total: useful diagnostic, not expected to always match.
priced = items.copy()
priced["item_net_eur"] = priced["quantity"] * priced["unit_price_eur"] * (1 - priced["discount_pct"] / 100)
item_net = priced.groupby("order_id")["item_net_eur"].sum()
order_review = sales[["order_id", "total_amount"]].copy()
order_review["items_net_eur"] = order_review["order_id"].map(item_net)
order_review["gap_eur"] = order_review["total_amount"] - order_review["items_net_eur"]
print("Eligible orders with > €0.01 item-total gap:", int(order_review["gap_eur"].abs().gt(0.01).sum()))
display(order_review.loc[order_review["gap_eur"].abs().gt(0.01)].sort_values("gap_eur", key=lambda s: s.abs(), ascending=False).head(10))

,campaign_name,base_after_budget,alternative_after_budget,difference_eur
campaign_id,,,,
C05,Spring Home Refresh,-315.85,132.82,448.67
C04,New Year Fitness,-6347.96,-6181.95,166.01
C12,Spring Clean 2026,2085.99,2132.44,46.45
C11,Black Friday 2025,-46838.21,-46800.52,37.69
C01,Back to Uni 2024,-10502.18,-10470.35,31.83
C10,Back to Uni 2025,-7347.53,-7323.48,24.05
C09,Summer Mega Sale,-194911.58,-194894.74,16.84
C02,Black Friday 2024,-41884.60,-41884.60,0.00
C03,Xmas Gifts 2024,3301.41,3301.41,0.00


Order-item duplicate rows: 97
These are scenario differences, not evidence that deleting item rows is correct.
Eligible orders with > €0.01 item-total gap: 177


,order_id,total_amount,items_net_eur,gap_eur
9814,O013138,1472.64,0.00,1472.64
13406,O012801,1331.06,-141.58,1472.64
2663,O000946,921.80,9.56,912.24
11427,O001999,879.03,0.00,879.03
9978,O003961,800.39,15.41,784.98
7476,O003840,577.20,-192.40,769.60
5983,O003096,1250.55,504.45,746.10
11733,O003542,755.56,9.46,746.10
13689,O014486,330.10,-305.70,635.80
11609,O012159,666.07,104.69,561.38


## 12. Mutation check — would our tests detect a common bug?

Deliberately introduce a wrong cost formula **in a separate fixture**, verify that the known-answer test rejects it, and leave real analysis data unchanged.

In [22]:
# Wrong: ignores quantities. The test MUST fail for the wrong formula.
wrong_fixture_cost = fi.groupby("order_id")["unit_cost_eur"].sum()
check_true("Mutation test detects omitted quantity multiplier",
           wrong_fixture_cost["O1"] != 80.0,
           f"Wrong formula would give €{wrong_fixture_cost['O1']:.2f} rather than €80.00")
print("A PASS here means the known-answer expectation distinguishes the incorrect formula.")

A PASS here means the known-answer expectation distinguishes the incorrect formula.


## 13. Final check report

**All PASS** means the arithmetic and exported results match these source tables. It does *not* validate unobserved attribution, actual ad spending, unit-cost accuracy, or incrementality.

In [23]:
results = pd.DataFrame(checks)
display(results)
failed = results[results["status"] == "FAIL"]
print(f"PASSED: {(results['status'] == 'PASS').sum()} / {len(results)}")
print(f"FAILED: {len(failed)}")
if not failed.empty:
    print("Investigate these checks before finalizing REPORT.md:")
    display(failed)
else:
    print("All numerical consistency checks passed.")

,check,actual,expected,difference,status
0,Unique order IDs,,True,NaN,PASS
1,Unique customer IDs,,True,NaN,PASS
2,Unique campaign IDs,,True,NaN,PASS
3,Unique product IDs,,True,NaN,PASS
4,Revenue-eligible statuses are completed,,True,NaN,PASS
...,...,...,...,...,...
56,Joining order costs preserves eligible row count,12811,12811,0.000000e+00,PASS
57,Campaign budgets strictly positive,,True,NaN,PASS
58,Sampled order costs match visible item-level a...,,True,NaN,PASS
59,Base cost matches main aggregation,3667297.18,3667297.18,4.656613e-10,PASS


PASSED: 61 / 61
FAILED: 0
All numerical consistency checks passed.


### Validation and robustness

All implemented calculation and consistency tests passed. A sensitivity analysis removing exact duplicate order-item rows showed that the main campaign-level conclusions remained unchanged. Spring Home Refresh was the exception, moving from slightly negative to slightly positive estimated contribution, and is therefore treated as approximately break-even.

However, 177 eligible orders had recorded totals that differed from calculated item values by more than €0.01. These discrepancies remain unresolved and may affect financial estimates. Recorded order totals were retained rather than silently corrected.

The analysis is reproducible and has passed its implemented validation checks, but source-data quality and marketing incrementality remain limitations.